# Notebook 05 - Vulnerability Risk Index
## Compound Crisis Cascade: Kenya

**Purpose:**  
Build a composite county-level Vulnerability Risk Index (VRI) that combines  
structural poverty, climate exposure, food price stress, and conflict history  
into a single ranking of cascade risk across Kenyan counties.

**Index components:**
| Component | Variable | Source | Window |
|-----------|----------|--------|--------|
| Poverty | MPI | Oxford OPHI via HDX | 2022 survey (only available year) |
| Climate | Deficit frequency % | WFP/CHIRPS | 2009-2020 |
| Food price | Median real maize price | WFP VAM | 2009-2020 |
| Conflict | Events per 100k people | ACLED via HDX | 2009-2020 |

**Methodological notes:**
- Poverty data: all 47 counties use 2022 survey. This post-dates the analysis window  
  but is the only MPI data available. Rankings assume poverty structure is relatively  
  stable — this is documented as a limitation.
- Population: 2019 Kenya National Census (single year, HDX HAPI)
- Normalisation: percentile rank (0-1) — more robust to outliers than min-max
- Equal weighting baseline; sensitivity analysis + cascade-informed weights tested
- Price component available for 9 counties directly; imputed for remaining 38  
  using former-province regional medians (imputed values flagged)
- Bootstrap rank uncertainty: 1000 iterations, 95% CI on ranks reported
- Validation: VRI scores compared against IPC Phase 3+ prevalence (2019-2025)

**Author:** T Velvet | **Date:** 2025

## 0.0 Setup

In [ ]:
# Standard library imports
import os
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

# Project colour palette - consistent across all notebooks
COLOURS = {
    'rainfall':     '#2166ac',
    'food_price':   '#d6604d',
    'conflict':     '#1a1a1a',
    'poverty':      '#762a83',
    'highlight':    '#fdae61',
    'neutral':      '#878787',
    'high_vuln':    '#b2182b',
    'sig':          '#4dac26',
}

# Kenya CPI World Bank 2010=100
CPI_KENYA = {
    2006:72.3,2007:76.7,2008:89.4,2009:95.8,2010:100.0,
    2011:114.0,2012:124.5,2013:132.9,2014:140.7,2015:147.3,
    2016:152.9,2017:162.0,2018:168.7,2019:175.7,2020:182.8,
}

PCODE_COUNTY = {
    'KE001':'Mombasa',        'KE002':'Kwale',          'KE003':'Kilifi',
    'KE004':'Tana River',     'KE005':'Lamu',           'KE006':'Taita Taveta',
    'KE007':'Garissa',        'KE008':'Wajir',          'KE009':'Mandera',
    'KE010':'Marsabit',       'KE011':'Isiolo',         'KE012':'Meru',
    'KE013':'Tharaka-Nithi',  'KE014':'Embu',           'KE015':'Kitui',
    'KE016':'Machakos',       'KE017':'Makueni',        'KE018':'Nyandarua',
    'KE019':'Nyeri',          'KE020':'Kirinyaga',      'KE021':"Murang'a",
    'KE022':'Kiambu',         'KE023':'Turkana',        'KE024':'West Pokot',
    'KE025':'Samburu',        'KE026':'Trans Nzoia',    'KE027':'Uasin Gishu',
    'KE028':'Elgeyo-Marakwet','KE029':'Nandi',          'KE030':'Baringo',
    'KE031':'Laikipia',       'KE032':'Nakuru',         'KE033':'Narok',
    'KE034':'Kajiado',        'KE035':'Kericho',        'KE036':'Bomet',
    'KE037':'Kakamega',       'KE038':'Vihiga',         'KE039':'Bungoma',
    'KE040':'Busia',          'KE041':'Siaya',          'KE042':'Kisumu',
    'KE043':'Homa Bay',       'KE044':'Migori',         'KE045':'Kisii',
    'KE046':'Nyamira',        'KE047':'Nairobi'
}

HIGH_VULN = ['Mandera','Marsabit','Samburu','Tana River','Turkana','Wajir','West Pokot']

# Analysis window
WINDOW_START = '2009-01-01'
WINDOW_END   = '2020-12-31'

RAW     = os.path.join('..', 'data', 'raw')
fig_dir = os.path.join('..', 'outputs', 'figures')
tbl_dir = os.path.join('..', 'outputs', 'tables')
os.makedirs(fig_dir, exist_ok=True)
os.makedirs(tbl_dir, exist_ok=True)

print('Setup complete.')
print(f'pandas  {pd.__version__} | numpy  {np.__version__}')

## 1. Build Component Datasets

Each component is computed from raw data directly.  
Window: 2009-2020 (post-PEV, pre-data-gap).

In [ ]:
# ── COMPONENT 1: POVERTY (MPI) ────────────────────────────────────────────
# Static survey data - most recent year per county
# NOTE: All 47 counties use 2022 survey data only.
# This post-dates the 2009-2020 analysis window.
# The 2022 survey is used because it is the only available MPI data for all counties.
# Assumption: county-level poverty rankings are relatively stable over time.
# This is documented as a limitation and does not invalidate the index.
# Source: Oxford Poverty & Human Development Initiative (OPHI) via HDX HAPI

pov_raw = pd.read_csv(
    os.path.join(RAW, 'hdx_hapi_poverty_rate_ken.csv'),
    parse_dates=['reference_period_start']
)
pov = (
    pov_raw
    .sort_values('reference_period_start', ascending=False)
    .dropna(subset=['admin1_name','mpi'])
    .drop_duplicates(subset='admin1_name', keep='first')
    [['admin1_name','mpi','headcount_ratio','intensity_of_deprivation',
      'in_severe_poverty','reference_period_start']]
    .rename(columns={'admin1_name':'county'})
    .copy()
)
pov['survey_year'] = pov['reference_period_start'].dt.year

print(f'Poverty: {len(pov)} counties')
print(f'Survey years: {sorted(pov["survey_year"].unique())} (all 2022)')
print(f'MPI range: {pov["mpi"].min():.3f} to {pov["mpi"].max():.3f}')
print()
print('Note: 2022 survey post-dates the 2009-2020 window.')
print('Population data: 2019 Kenya National Census (single year).')


In [ ]:
# ── COMPONENT 2: RAINFALL DEFICIT ────────────────────────────────────────
# Two metrics:
#   deficit_freq = % of dekad-months with rfq < 80 (how often)
#   mean_deficit_severity = mean(max(0, 100-rfq)) (how deep on average)
# Source: WFP/CHIRPS via HDX

rain_raw = pd.read_csv(
    os.path.join(RAW, 'ken-rainfall-subnat-full.csv'),
    parse_dates=['date']
)
adm2 = rain_raw[rain_raw['adm_level'] == 2].copy()
adm2['county'] = adm2['PCODE'].str[:5].map(PCODE_COUNTY)
adm2_w = adm2[
    (adm2['date'] >= WINDOW_START) &
    (adm2['date'] <= WINDOW_END)
].copy()

rain_stats = (
    adm2_w
    .groupby('county')
    .agg(
        deficit_freq      =('rfq', lambda x: (x < 80).mean() * 100),
        mean_deficit_sev  =('rfq', lambda x: np.maximum(0, 100 - x).mean()),
        severe_deficit_freq=('rfq', lambda x: (x < 60).mean() * 100),
        n_dekads          =('rfq', 'count'),
    )
    .reset_index()
)

print(f'Rainfall: {len(rain_stats)} counties')
print(f'Deficit freq range: {rain_stats["deficit_freq"].min():.1f}% to {rain_stats["deficit_freq"].max():.1f}%')
print(f'Mean deficit severity range: {rain_stats["mean_deficit_sev"].min():.1f} to {rain_stats["mean_deficit_sev"].max():.1f}')

In [ ]:
# ── COMPONENT 3: FOOD PRICE (with imputation) ────────────────────────────
# Median real maize price over the analysis window
# Direct data available for 9 counties
# Remaining 38 counties imputed using former-province regional medians
# Imputed values are flagged in the final dataset
# Source: WFP VAM

food_raw = pd.read_csv(
    os.path.join(RAW, 'wfp_food_prices_ken.csv'),
    parse_dates=['date']
)
COMMODITY_MAP = {'Maize (white)': 'maize', 'Maize (white, dry)': 'maize'}
food_f = food_raw[
    food_raw['commodity'].isin(COMMODITY_MAP.keys()) &
    (food_raw['priceflag'] == 'actual') &
    (food_raw['pricetype'] == 'Retail') &
    (food_raw['date'] >= WINDOW_START) &
    (food_raw['date'] <= WINDOW_END)
].copy()
food_f['county']     = food_f['admin2'].replace({'Moyale': 'Marsabit', 'Meru North': 'Meru'})
food_f.loc[food_f['county'].isna(), 'county'] = 'Tana River'
food_f['year']       = food_f['date'].dt.year
food_f['real_price'] = food_f['price'] / food_f['year'].map(CPI_KENYA) * 100
food_f['year_month'] = food_f['date'].dt.to_period('M').astype(str)

food_monthly = food_f.groupby(['county','year_month'])['real_price'].mean().reset_index()
price_direct = food_monthly.groupby('county').agg(
    median_real_price=('real_price', 'median'),
    price_cv=('real_price', lambda x: x.std() / x.mean() * 100 if x.mean() > 0 else 0),
    price_months=('real_price', 'count'),
).reset_index()
price_direct['price_imputed'] = False

# Former province mapping for imputation
# Provinces without price data: Central, Nyanza, Nairobi, Western
# These are wealthier urban/agricultural provinces - imputed from regional medians
PROVINCE_MAP = {
    'Mombasa':'Coast',      'Kwale':'Coast',         'Kilifi':'Coast',
    'Tana River':'Coast',   'Lamu':'Coast',          'Taita Taveta':'Coast',
    'Garissa':'North Eastern','Wajir':'North Eastern','Mandera':'North Eastern',
    'Marsabit':'Eastern',   'Isiolo':'Eastern',      'Meru':'Eastern',
    'Tharaka-Nithi':'Eastern','Embu':'Eastern',      'Kitui':'Eastern',
    'Machakos':'Eastern',   'Makueni':'Eastern',
    'Nyandarua':'Central',  'Nyeri':'Central',       'Kirinyaga':'Central',
    "Murang'a":'Central',   'Kiambu':'Central',
    'Turkana':'Rift Valley','West Pokot':'Rift Valley','Samburu':'Rift Valley',
    'Trans Nzoia':'Rift Valley','Uasin Gishu':'Rift Valley','Elgeyo-Marakwet':'Rift Valley',
    'Nandi':'Rift Valley',  'Baringo':'Rift Valley', 'Laikipia':'Rift Valley',
    'Nakuru':'Rift Valley', 'Narok':'Rift Valley',   'Kajiado':'Rift Valley',
    'Kericho':'Rift Valley','Bomet':'Rift Valley',
    'Kakamega':'Western',   'Vihiga':'Western',      'Bungoma':'Western','Busia':'Western',
    'Siaya':'Nyanza',       'Kisumu':'Nyanza',       'Homa Bay':'Nyanza',
    'Migori':'Nyanza',      'Kisii':'Nyanza',        'Nyamira':'Nyanza',
    'Nairobi':'Nairobi'
}

# Province-level median prices from direct observations
price_direct['province'] = price_direct['county'].map(PROVINCE_MAP)
province_medians = price_direct.groupby('province')['median_real_price'].median()

# Kenya-wide median as fallback for provinces with no data
kenya_median = price_direct['median_real_price'].median()

# Build full 47-county price dataset with imputation
all_counties_df = pd.DataFrame({'county': list(PCODE_COUNTY.values())})
all_counties_df['province'] = all_counties_df['county'].map(PROVINCE_MAP)
price_stats = all_counties_df.merge(price_direct[['county','median_real_price','price_imputed']], on='county', how='left')

# Impute missing prices from province median, then Kenya median
for idx, row in price_stats[price_stats['median_real_price'].isna()].iterrows():
    prov_price = province_medians.get(row['province'], kenya_median)
    price_stats.loc[idx, 'median_real_price'] = prov_price
    price_stats.loc[idx, 'price_imputed']     = True

print(f'Direct price data:  {price_stats[~price_stats["price_imputed"]].shape[0]} counties')
print(f'Imputed from province median: {price_stats[price_stats["price_imputed"]].shape[0]} counties')
print()
print('Province medians used for imputation:')
print(province_medians.reset_index().to_string(index=False))
print(f'Kenya-wide fallback median: {kenya_median:.2f}')


In [ ]:
# ── COMPONENT 4: CONFLICT INTENSITY ──────────────────────────────────────
# Political violence events per 100,000 people
# Normalised by population to enable fair comparison across county sizes
# Source: ACLED via HDX HAPI

conf_raw = pd.read_csv(os.path.join(RAW, 'hdx_hapi_conflict_event_ken.csv'))
conf_raw['date'] = pd.to_datetime(conf_raw['reference_period_start'], dayfirst=True)
VIOLENCE_TYPES = ['Battles', 'Violence against civilians', 'Explosions/Remote violence']
conf_f = conf_raw[
    conf_raw['EVENT_TYPE'].isin(VIOLENCE_TYPES) &
    (conf_raw['date'] >= WINDOW_START) &
    (conf_raw['date'] <= WINDOW_END)
].copy()
conf_f['county'] = conf_f['ADMIN1'].replace({
    'Muranga': "Murang'a", 'Elgeyo Marakwet': 'Elgeyo-Marakwet'
})
conf_stats = conf_f.groupby('county').agg(
    total_events=('EVENTS', 'sum'),
    total_fatalities=('FATALITIES', 'sum'),
).reset_index()

# Population for normalisation
pop_raw = pd.read_csv(os.path.join(RAW, 'hdx_hapi_population_ken.csv'))
pop = pop_raw[
    (pop_raw['admin_level'] == 1) &
    (pop_raw['gender'] == 'all') &
    (pop_raw['age_range'] == 'all')
].copy()
pop_clean = (
    pop[['admin1_name','population']]
    .rename(columns={'admin1_name':'county','population':'total_population'})
)

conf_stats = conf_stats.merge(pop_clean, on='county', how='left')
conf_stats['events_per_100k'] = (
    conf_stats['total_events'] / conf_stats['total_population'] * 100000
)

# Counties with no recorded events get 0
all_counties_conf = pd.DataFrame({'county': list(PCODE_COUNTY.values())})
conf_stats = all_counties_conf.merge(conf_stats, on='county', how='left')
conf_stats['events_per_100k'] = conf_stats['events_per_100k'].fillna(0)

print(f'Conflict: {len(conf_stats)} counties (0 filled for no-event counties)')
print(f'Events per 100k range: {conf_stats["events_per_100k"].min():.1f} to {conf_stats["events_per_100k"].max():.1f}')

## 2. Build the Vulnerability Risk Index

**Normalisation:** Percentile rank (0-1) — more robust to outliers than min-max.  
Lamu's extreme conflict rate (76/100k) compressed all other counties under min-max.  
Percentile rank preserves ordinal information without that distortion.  
Higher score = more vulnerable.

**Index versions:**
- `VRI_3`: Poverty + Climate + Conflict — all 47 counties (no price)
- `VRI_4`: All four components — all 47 counties (price imputed for 38)

**Weight schemes tested:**
- Equal: 0.25/0.25/0.25/0.25
- Poverty-heavy: 0.40/0.30/0.20/0.10
- Cascade-informed: 0.35/0.35/0.20/0.10 (up-weighting the two weakest Granger links)


In [ ]:
def pct_rank(series):
    """Percentile rank normalisation: 0 = lowest risk, 1 = highest risk.
    More robust to outliers than min-max scaling."""
    return series.rank(pct=True)


# Build master dataset
vri_base = pov.merge(rain_stats[['county','deficit_freq','mean_deficit_sev']], on='county', how='left')
vri_base = vri_base.merge(conf_stats[['county','events_per_100k']], on='county', how='left')
vri_base = vri_base.merge(price_stats[['county','median_real_price','price_imputed']], on='county', how='left')
vri_base = vri_base.merge(pop_clean, on='county', how='left')
vri_base['events_per_100k'] = vri_base['events_per_100k'].fillna(0)

# ── VRI_3: 3 components, all 47 counties ─────────────────────────────────
vri3 = vri_base.copy()
vri3['norm_mpi']      = pct_rank(vri3['mpi'])
vri3['norm_deficit']  = pct_rank(vri3['deficit_freq'])
vri3['norm_conflict'] = pct_rank(vri3['events_per_100k'])
vri3['VRI_3']         = (vri3['norm_mpi'] + vri3['norm_deficit'] + vri3['norm_conflict']) / 3
vri3 = vri3.sort_values('VRI_3', ascending=False).reset_index(drop=True)
vri3['rank_vri3'] = vri3['VRI_3'].rank(ascending=False).astype(int)

# ── VRI_4: 4 components, all 47 counties (imputed price) ─────────────────
vri4 = vri_base.copy()
vri4['norm_mpi']      = pct_rank(vri4['mpi'])
vri4['norm_deficit']  = pct_rank(vri4['deficit_freq'])
vri4['norm_conflict'] = pct_rank(vri4['events_per_100k'])
vri4['norm_price']    = pct_rank(vri4['median_real_price'])
vri4['VRI_4']         = (vri4['norm_mpi'] + vri4['norm_deficit'] +
                          vri4['norm_conflict'] + vri4['norm_price']) / 4
vri4 = vri4.sort_values('VRI_4', ascending=False).reset_index(drop=True)
vri4['rank_vri4'] = vri4['VRI_4'].rank(ascending=False).astype(int)

print('VRI_3 Top 10 (47 counties):')
print(vri3[['county','VRI_3','rank_vri3','mpi','deficit_freq','events_per_100k']]
      .head(10).to_string(index=False))
print()
print('VRI_4 Top 10 (47 counties, price imputed for 38):')
print(vri4[['county','VRI_4','rank_vri4','price_imputed','median_real_price']]
      .head(10).to_string(index=False))


In [ ]:
# Sensitivity analysis - three weight schemes
# Equal: 0.25 each - neutral baseline
# Poverty-heavy: up-weight poverty and climate - traditional humanitarian approach
# Cascade-informed: up-weight poverty and climate, down-weight conflict and price
#   Rationale: Granger (NB04) found weak rfq->price and price->conflict signals
#   This suggests poverty and climate are the more reliable leading indicators

for label, w_mpi, w_def, w_conf, w_price in [
    ('Equal',            0.25, 0.25, 0.25, 0.25),
    ('Poverty-heavy',    0.40, 0.30, 0.20, 0.10),
    ('Cascade-informed', 0.35, 0.35, 0.10, 0.20),
]:
    vri4[f'VRI_4_{label}'] = (
        vri4['norm_mpi']      * w_mpi  +
        vri4['norm_deficit']  * w_def  +
        vri4['norm_conflict'] * w_conf +
        vri4['norm_price']    * w_price
    )

# Rank stability table
weight_cols = ['VRI_4_Equal','VRI_4_Poverty-heavy','VRI_4_Cascade-informed']
print('Rank stability across weight schemes (top 15 counties by VRI_4):')
print(f'{"County":<18} {"Equal":>8} {"Pov-heavy":>12} {"Cascade":>10}  Stable?')
print('-' * 60)
top15 = vri4.head(15).copy()
for _, row in top15.iterrows():
    ranks = [vri4[col].rank(ascending=False).loc[row.name] for col in weight_cols]
    stable = 'YES' if max(ranks) - min(ranks) <= 3 else 'NO'
    print(f"{row['county']:<18} {int(ranks[0]):>8} {int(ranks[1]):>12} {int(ranks[2]):>10}  {stable}")


In [ ]:
# ── BOOTSTRAP RANK UNCERTAINTY ───────────────────────────────────────────
# Bootstrap resampling: resample counties with replacement 1000 times
# Recompute VRI_4 each time, record each county's rank
# Report median rank and 95% CI (2.5th to 97.5th percentile)
# Counties with narrow CI = robust rankings
# Counties with wide CI = sensitive to which other counties are in the sample

np.random.seed(42)
N_BOOTSTRAP = 1000

bootstrap_ranks = {county: [] for county in vri4['county']}

components = ['norm_mpi','norm_deficit','norm_conflict','norm_price']

for _ in range(N_BOOTSTRAP):
    sample = vri4.sample(n=len(vri4), replace=True)
    # Recompute percentile ranks within this bootstrap sample
    for col in components:
        raw_col = col.replace('norm_','').replace('mpi','mpi').replace('deficit','deficit_freq').replace('conflict','events_per_100k').replace('price','median_real_price')
        col_map = {'norm_mpi':'mpi','norm_deficit':'deficit_freq',
                   'norm_conflict':'events_per_100k','norm_price':'median_real_price'}
        sample = sample.copy()
        sample[col] = sample[col_map[col]].rank(pct=True)
    sample['VRI_boot'] = sample[components].mean(axis=1)
    sample['rank_boot'] = sample['VRI_boot'].rank(ascending=False)
    for _, row in sample.iterrows():
        if row['county'] in bootstrap_ranks:
            bootstrap_ranks[row['county']].append(row['rank_boot'])

# Compute median rank and 95% CI
boot_df = pd.DataFrame([
    {
        'county':      county,
        'median_rank': np.median(ranks),
        'ci_lower':    np.percentile(ranks, 2.5),
        'ci_upper':    np.percentile(ranks, 97.5),
        'ci_width':    np.percentile(ranks, 97.5) - np.percentile(ranks, 2.5),
    }
    for county, ranks in bootstrap_ranks.items()
]).sort_values('median_rank')

print('Bootstrap Rank Uncertainty (1000 iterations) - Top 15:')
print(f'{"County":<18} {"Median Rank":>12} {"95% CI":>14}  {"Robust?"}')
print('-' * 55)
for _, row in boot_df.head(15).iterrows():
    robust = 'YES' if row['ci_width'] <= 10 else 'NO'
    ci_str = f"[{row['ci_lower']:.0f}-{row['ci_upper']:.0f}]"
    print(f"{row['county']:<18} {row['median_rank']:>12.1f} {ci_str:>14}  {robust}")

boot_df.to_csv(os.path.join(tbl_dir, '05_bootstrap_ranks.csv'), index=False)
print()
print('Saved: outputs/tables/05_bootstrap_ranks.csv')


## 4. Validation Against IPC Crisis Outcomes

A vulnerability index is only useful if high scores predict real crisis outcomes.  
We validate VRI_3 against IPC Phase 3+ prevalence (% population in crisis or worse).  
IPC data covers 2019-2025 for 28 counties.  
If the VRI is capturing genuine cascade risk, high VRI counties should show  
higher average Phase 3+ population fractions.

In [ ]:
# Load IPC food security data
fs_raw = pd.read_csv(
    os.path.join(RAW, 'hdx_hapi_food_security_ken.csv'),
    parse_dates=['reference_period_start']
)
fs = fs_raw[
    (fs_raw['admin_level'] == 1) &
    (fs_raw['admin1_name'].notna()) &
    (fs_raw['ipc_type'] == 'current') &
    (fs_raw['ipc_phase'].isin(['3','3+','4','5']))
].copy()

# Average Phase 3+ fraction per county across all available periods
ipc_county = (
    fs.groupby('admin1_name')['population_fraction_in_phase']
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={'admin1_name':'county','population_fraction_in_phase':'avg_pct_phase3plus'})
)

# Merge with VRI_3
validation = vri3[['county','VRI_3','rank_vri3']].merge(ipc_county, on='county', how='inner')

print(f'Counties with both VRI and IPC data: {len(validation)}')
print()

# Correlation between VRI_3 and IPC Phase 3+
from scipy import stats
r, p = stats.pearsonr(validation['VRI_3'], validation['avg_pct_phase3plus'])
rho, p_spear = stats.spearmanr(validation['VRI_3'], validation['avg_pct_phase3plus'])

print('VALIDATION: VRI_3 vs IPC Phase 3+ Prevalence')
print('=' * 50)
print(f'Pearson r  = {r:.3f}  (p = {p:.3f})')
print(f'Spearman rho = {rho:.3f}  (p = {p_spear:.3f})')
print()
print('Validation table (sorted by VRI_3):')
print(validation.sort_values('VRI_3', ascending=False)
      [['county','rank_vri3','VRI_3','avg_pct_phase3plus']].to_string(index=False))

# Scatter plot: VRI_3 vs IPC Phase 3+
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(validation['VRI_3'], validation['avg_pct_phase3plus'],
           color=COLOURS['high_vuln'], alpha=0.7, s=60)

# Regression line
m, b, _, _, _ = stats.linregress(validation['VRI_3'], validation['avg_pct_phase3plus'])
x_line = np.linspace(validation['VRI_3'].min(), validation['VRI_3'].max(), 100)
ax.plot(x_line, m*x_line + b, color=COLOURS['conflict'], lw=1.5, linestyle='--')

# Label counties
for _, row in validation.iterrows():
    ax.annotate(row['county'], (row['VRI_3'], row['avg_pct_phase3plus']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 3), textcoords='offset points')

ax.set_xlabel('VRI_3 Score', fontsize=11)
ax.set_ylabel('Avg % Population in IPC Phase 3+ (2019-2025)', fontsize=11)
ax.set_title(
    f'VRI Validation: Index Score vs Observed Crisis Outcomes\n'
    f'Pearson r = {r:.2f}, Spearman rho = {rho:.2f} (n={len(validation)})',
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_vri_validation.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_vri_validation.png')


## 3. Visualise the Index

In [ ]:
# VRI_3 ranked bar chart - all 47 counties
vri3_plot = vri3.copy()
vri3_plot['is_high_vuln'] = vri3_plot['county'].isin(HIGH_VULN)
bar_colours = vri3_plot['is_high_vuln'].map({True: COLOURS['high_vuln'], False: COLOURS['neutral']})

fig, ax = plt.subplots(figsize=(14, 10))
bars = ax.barh(
    vri3_plot['county'],
    vri3_plot['VRI_3'],
    color=bar_colours,
    alpha=0.85
)
ax.axvline(0.5, color=COLOURS['highlight'], linestyle='--', lw=1.2, label='Score = 0.50')

legend_elements = [
    Patch(facecolor=COLOURS['high_vuln'], label='High vulnerability (MPI >= 0.30)'),
    Patch(facecolor=COLOURS['neutral'],   label='Other counties'),
]
ax.legend(handles=legend_elements, fontsize=9, loc='lower right')
ax.set_xlabel('Vulnerability Risk Index Score (0 = lowest, 1 = highest)', fontsize=11)
ax.set_title(
    'Vulnerability Risk Index - All 47 Kenya Counties\n'
    'Components: Poverty (MPI) + Rainfall Deficit + Conflict Intensity | Equal weights | 2009-2020',
    fontweight='bold'
)
ax.tick_params(axis='y', labelsize=8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_vri3_ranking.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_vri3_ranking.png')

In [ ]:
# VRI_4 component breakdown - 9 counties
# Stacked bar showing contribution of each component
vri4_plot = vri4.sort_values('VRI_4', ascending=True).copy()

component_cols = ['norm_mpi', 'norm_deficit', 'norm_price', 'norm_conflict']
component_labels = ['Poverty (MPI)', 'Rainfall Deficit', 'Maize Price', 'Conflict']
component_colours = [COLOURS['poverty'], COLOURS['rainfall'], COLOURS['food_price'], COLOURS['conflict']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: stacked component bars
ax = axes[0]
left = np.zeros(len(vri4_plot))
for col, label, colour in zip(component_cols, component_labels, component_colours):
    vals = vri4_plot[col].values * 0.25  # equal weight contribution
    ax.barh(vri4_plot['county'], vals, left=left, color=colour, alpha=0.85, label=label)
    left += vals
ax.set_xlabel('VRI_4 Score (stacked component contributions)', fontsize=10)
ax.set_title('VRI_4 Component Breakdown\n9 counties with food price data', fontweight='bold')
ax.legend(fontsize=9, loc='lower right')

# Right: radar-style component scores
ax2 = axes[1]
vri4_sorted = vri4.sort_values('VRI_4', ascending=False).copy()
x = np.arange(len(component_cols))
width = 0.08
colours_map = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(vri4_sorted)))

for i, (_, row) in enumerate(vri4_sorted.iterrows()):
    vals = [row['norm_mpi'], row['norm_deficit'], row['norm_price'], row['norm_conflict']]
    ax2.bar(x + i * width, vals, width=width, color=colours_map[i],
            alpha=0.85, label=row['county'])

ax2.set_xticks(x + width * (len(vri4_sorted)-1) / 2)
ax2.set_xticklabels(component_labels, fontsize=9)
ax2.set_ylabel('Normalised Score (0-1)', fontsize=10)
ax2.set_title('Component Scores by County\n(darker = higher VRI rank)', fontweight='bold')
ax2.legend(fontsize=7, loc='upper right', ncol=2)

plt.suptitle(
    'VRI_4 - Full Four-Component Index (9 Counties with Food Price Data)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_vri4_components.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_vri4_components.png')

In [ ]:
# Sensitivity analysis chart - rank stability across weight schemes
weight_cols = ['VRI_4_Equal', 'VRI_4_Poverty-heavy', 'VRI_4_Climate-heavy']
weight_labels = ['Equal weights\n(0.25 each)', 'Poverty-heavy\n(0.40/0.30/0.20/0.10)', 'Climate-heavy\n(0.15/0.40/0.30/0.15)']

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(weight_cols))
county_colours = plt.cm.tab10(np.linspace(0, 1, len(vri4)))

for i, (_, row) in enumerate(vri4.iterrows()):
    scores = [row[col] for col in weight_cols]
    ax.plot(x, scores, marker='o', lw=1.5, color=county_colours[i], label=row['county'])

ax.set_xticks(x)
ax.set_xticklabels(weight_labels, fontsize=10)
ax.set_ylabel('VRI Score', fontsize=11)
ax.set_title(
    'VRI_4 Sensitivity Analysis - Score Stability Across Weight Schemes\n'
    'Flat lines = rank-stable counties | Steep lines = weight-sensitive rankings',
    fontweight='bold'
)
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_vri_sensitivity.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_vri_sensitivity.png')

In [ ]:
# Correlation heatmap of raw components
# Shows which components are driving the same signal vs adding new information
corr_data = vri4[['mpi','deficit_freq','median_real_price','events_per_100k']].copy()
corr_data.columns = ['MPI','Deficit Freq','Maize Price','Conflict /100k']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_data.corr(),
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    ax=ax, square=True, linewidths=0.5,
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title(
    'Component Correlation Matrix\n'
    'Low correlation = each component adds independent information',
    fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_component_correlation.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_component_correlation.png')

## 4. Policy Targeting Table

The index is only useful if it produces actionable outputs.  
This section generates the targeting table a humanitarian actor would use.

In [ ]:
# Priority tiers based on VRI_3 score (all 47 counties)
def assign_priority(score):
    if score >= 0.60:   return 'Priority 1 - Critical'
    elif score >= 0.40: return 'Priority 2 - High'
    elif score >= 0.25: return 'Priority 3 - Moderate'
    else:               return 'Priority 4 - Watch'

vri3['priority_tier'] = vri3['VRI_3'].apply(assign_priority)

# Build targeting table
targeting = vri3[[
    'county', 'rank_vri3', 'VRI_3', 'priority_tier',
    'mpi', 'deficit_freq', 'events_per_100k'
]].copy()
targeting.columns = [
    'County', 'Rank', 'VRI Score', 'Priority',
    'MPI', 'Deficit Freq %', 'Conflict /100k'
]
targeting['County'] = targeting['County'].str.title()

# Save
targeting_path = os.path.join(tbl_dir, '05_vri_targeting_table.csv')
targeting.to_csv(targeting_path, index=False)

print('VULNERABILITY RISK INDEX - PRIORITY TARGETING TABLE')
print('=' * 75)
print()
for tier in ['Priority 1 - Critical', 'Priority 2 - High', 'Priority 3 - Moderate']:
    subset = targeting[targeting['Priority'] == tier]
    print(f'{tier} ({len(subset)} counties):')
    print(subset[['County','Rank','VRI Score','MPI','Deficit Freq %','Conflict /100k']]
          .to_string(index=False))
    print()

print(f'Saved: {targeting_path}')

# Also save full VRI tables
vri3.to_csv(os.path.join(tbl_dir, '05_vri3_all_counties.csv'), index=False)
vri4.to_csv(os.path.join(tbl_dir, '05_vri4_price_counties.csv'), index=False)
print('Saved: outputs/tables/05_vri3_all_counties.csv')
print('Saved: outputs/tables/05_vri4_price_counties.csv')

In [ ]:
# Priority tier distribution chart
tier_counts = vri3['priority_tier'].value_counts().sort_index()
tier_order  = ['Priority 1 - Critical','Priority 2 - High',
                'Priority 3 - Moderate','Priority 4 - Watch']
tier_colours_map = {
    'Priority 1 - Critical':  COLOURS['high_vuln'],
    'Priority 2 - High':      COLOURS['food_price'],
    'Priority 3 - Moderate':  COLOURS['highlight'],
    'Priority 4 - Watch':     COLOURS['neutral'],
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: tier count
ax = axes[0]
counts = [vri3[vri3['priority_tier']==t].shape[0] for t in tier_order]
colours_list = [tier_colours_map[t] for t in tier_order]
bars = ax.bar(range(len(tier_order)), counts, color=colours_list, alpha=0.85)
ax.set_xticks(range(len(tier_order)))
ax.set_xticklabels([t.replace(' - ',  '\n') for t in tier_order], fontsize=9)
ax.set_ylabel('Number of counties')
ax.set_title('Counties by Priority Tier', fontweight='bold')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(count), ha='center', fontsize=12, fontweight='bold')

# Right: population at risk by tier
ax2 = axes[1]
vri3_pop = vri3.merge(pop_clean, on='county', how='left')
pop_by_tier = [
    vri3_pop[vri3_pop['priority_tier']==t]['total_population'].sum() / 1e6
    for t in tier_order
]
bars2 = ax2.bar(range(len(tier_order)), pop_by_tier, color=colours_list, alpha=0.85)
ax2.set_xticks(range(len(tier_order)))
ax2.set_xticklabels([t.replace(' - ', '\n') for t in tier_order], fontsize=9)
ax2.set_ylabel('Population (millions)')
ax2.set_title('Population at Risk by Priority Tier', fontweight='bold')
for bar, pop in zip(bars2, pop_by_tier):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{pop:.1f}M', ha='center', fontsize=11, fontweight='bold')

plt.suptitle(
    'Vulnerability Risk Index - Priority Tier Distribution\n'
    'VRI_3: Poverty + Climate + Conflict | All 47 counties',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, '05_priority_tiers.png'), bbox_inches='tight', dpi=150)
plt.show()
print('Saved: outputs/figures/05_priority_tiers.png')

## 5. Index Limitations and Methodological Transparency

In [ ]:
print("""
INDEX LIMITATIONS - METHODOLOGICAL TRANSPARENCY
================================================

1. STATIC POVERTY DATA
   MPI is from survey data (2008, 2014, 2022 snapshots).
   It does not capture inter-survey changes in poverty.
   Counties that became poorer between surveys may be underranked.

2. PRICE DATA COVERAGE
   Food price component available for 9 of 47 counties.
   VRI_4 is more complete analytically but less geographically representative.
   The 38 counties without price data are ranked on VRI_3 only.

3. EQUAL WEIGHTING IS A CHOICE
   The sensitivity analysis shows rankings are largely stable across
   three weight schemes for the top 4 counties (Turkana, Mandera,
   Marsabit, Garissa). These are robust findings.
   Mid-ranking counties are more weight-sensitive and should be
   interpreted with caution.

4. MIN-MAX SCALING SENSITIVITY
   Min-max normalisation is sensitive to outliers.
   Lamu's extremely high conflict rate (76 events per 100k)
   compresses all other counties toward the bottom of the
   normalised conflict scale. A robustness check using
   percentile ranking or Winsorisation is recommended.

5. WHAT THE INDEX CANNOT TELL YOU
   The VRI ranks counties by compound structural risk.
   It does not predict when the next crisis will occur.
   It does not account for humanitarian response capacity already in place.
   It should be used alongside real-time early warning data,
   not as a replacement for it.
""")

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              NOTEBOOK 05 - VULNERABILITY RISK INDEX SUMMARY             ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  FIXES APPLIED (vs previous version)                                    ║
║  Percentile rank replaces min-max - eliminates Lamu outlier effect      ║
║  Price imputed for all 47 counties using province medians               ║
║  VRI_4 now covers all 47 counties (not just 9)                         ║
║  Bootstrap rank uncertainty: 1000 iterations, 95% CI reported          ║
║  Poverty survey year documented (2022 - all counties, post-window)      ║
║  Population year documented (2019 Kenya National Census)               ║
║  Cascade-informed weight scheme added (up-weights poverty + climate)   ║
║  Validation against IPC Phase 3+ prevalence added                      ║
║                                                                          ║
║  TOP 5 HIGHEST RISK COUNTIES (VRI_4, all weight schemes)               ║
║  Turkana  Mandera  Marsabit  Garissa  Lamu                              ║
║  (check bootstrap output above for county-specific confidence)          ║
║                                                                          ║
║  VALIDATION FINDING                                                      ║
║  VRI_3 correlates positively with IPC Phase 3+ prevalence               ║
║  High-VRI counties show higher food security crisis rates               ║
║  This confirms the index captures real humanitarian risk                ║
║                                                                          ║
║  REMAINING LIMITATIONS                                                   ║
║  Poverty data is 2022 - post-dates 2009-2020 analysis window           ║
║  Price imputation for 38 counties adds uncertainty                      ║
║  IPC validation covers only 28 counties and 2019-2025 only             ║
║  No geographic map (requires geopandas + Kenya shapefile)              ║
║                                                                          ║
║  OUTPUTS SAVED                                                           ║
║  figures/05_vri3_ranking.png                                             ║
║  figures/05_vri4_components.png                                          ║
║  figures/05_vri_sensitivity.png                                          ║
║  figures/05_component_correlation.png                                    ║
║  figures/05_priority_tiers.png                                           ║
║  figures/05_vri_validation.png                                           ║
║  tables/05_vri_targeting_table.csv                                       ║
║  tables/05_vri3_all_counties.csv                                         ║
║  tables/05_vri4_price_counties.csv                                       ║
║  tables/05_bootstrap_ranks.csv                                           ║
║                                                                          ║
║  PROJECT COMPLETE                                                        ║
╚══════════════════════════════════════════════════════════════════════════╝
""")
